## 1. Setup Environment

In [ ]:
# Clone repository
!git clone -b new-cross-moe https://github.com/glucose20/Temp.git
%cd Temp

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install pandas numpy scipy scikit-learn tqdm

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# ⚙️ CONFIGURE YOUR EXPERIMENT HERE
DATASET = "davis"       # davis, kiba, metz
RUNNING_SET = "novel-pair"  # warm, novel-drug, novel-prot, novel-pair
FOLD = 0                # 0-4
EPOCHS = 100            # Number of training epochs
QUICK_TEST = False      # Set True for quick test (only 2 configs)

## 3. Run A/B Testing

In [ ]:
# Run full A/B testing
quick_flag = "--quick" if QUICK_TEST else ""
!python scripts/ab_testing_colab.py \
    --dataset {DATASET} \
    --running_set {RUNNING_SET} \
    --fold {FOLD} \
    --epochs {EPOCHS} \
    {quick_flag}

## 4. View Results

In [ ]:
import pandas as pd
import os
from glob import glob

# Find latest results
result_dirs = sorted(glob("ab_results/*"))
if result_dirs:
    latest_dir = result_dirs[-1]
    summary_file = os.path.join(latest_dir, "summary.csv")
    
    if os.path.exists(summary_file):
        df = pd.read_csv(summary_file)
        print("\n📊 Results Summary:")
        display(df)
        
        # Highlight best
        if 'mse' in df.columns:
            best_mse_idx = df['mse'].idxmin()
            print(f"\n🏆 Best MSE: {df.loc[best_mse_idx, 'config']} ({df.loc[best_mse_idx, 'mse']:.6f})")
        if 'ci' in df.columns:
            best_ci_idx = df['ci'].idxmax()
            print(f"🏆 Best CI:  {df.loc[best_ci_idx, 'config']} ({df.loc[best_ci_idx, 'ci']:.6f})")
else:
    print("No results found. Run the A/B testing first.")

## 5. Run Individual Experiment (Optional)

If you want to run a specific configuration:

In [ ]:
# Example: Run specific MoE configuration
# Uncomment and modify as needed

# !python code/train.py \
#     --fold 0 \
#     --dataset davis \
#     --running_set novel-pair \
#     --epochs 100 \
#     --num_experts 4 \
#     --top_k 2 \
#     --load_balance_weight 0.01 \
#     --moe_noise_std 0.1 \
#     --no_wandb

## 6. Download Results

In [ ]:
# Zip and download results
from google.colab import files
import shutil

if result_dirs:
    latest_dir = result_dirs[-1]
    zip_name = f"ab_results_{os.path.basename(latest_dir)}"
    shutil.make_archive(zip_name, 'zip', latest_dir)
    files.download(f"{zip_name}.zip")
    print(f"📥 Downloaded: {zip_name}.zip")